In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
#--- 1. Load Data ---
df = pd.read_csv('Datasets/stations_full.csv', skiprows=[1])

In [3]:
#--- 2. Preprocessing ---
for col in df.columns:
    if col not in ['station_code', 'monitoring_location', 'state_name']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

target_features = ['do_max', 'bod_max', 'fecal_coliform_max']
df.dropna(subset=target_features, inplace=True)

In [4]:
#--- 3. Feature Engineering: Create Target Variable (WQI Category) ---
def classify_wqi(row):
    do_max = row['do_max']
    bod_max = row['bod_max']
    fecal_coliform_max = row['fecal_coliform_max']

    if do_max > 6.5 and bod_max < 2 and fecal_coliform_max < 100:
        return 'Good'
    elif 5 < do_max <= 6.5 and 2 <= bod_max < 5 and 100 <= fecal_coliform_max < 1000:
        return 'Moderate'
    else:
        return 'Poor'

df['wqi_category'] = df.apply(classify_wqi, axis=1)

In [5]:
#--- 4. Prepare Features (X) and Target (y) ---
if df['wqi_category'].nunique() < 2:
    print("The created target variable 'wqi_category' has only one class.")
    print("Cannot train a classifier. Please adjust the classification logic in classify_wqi function.")
    exit()

features = [col for col in df.columns if '_min' in col or '_max' in col] + ['state_name']
X = df[features]
y = df['wqi_category']

for col in X.select_dtypes(include=['number']).columns:
    X[col].fillna(X[col].mean(), inplace=True)

if 'state_name' in X.columns:
    le = LabelEncoder()
    X['state_name'] = le.fit_transform(X['state_name'].astype(str))

/tmp/ipykernel_182866/3486443313.py:12: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  X[col].fillna(X[col].mean(), inplace=True)


In [6]:
#--- 5. Feature Scaling ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
#--- 6. Train the Model ---
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

model = GaussianNB(var_smoothing=1e-9)

model.fit(X_train, y_train)

ValueError: Input X contains NaN.
GaussianNB does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
#--- 7. Evaluate the Model ---
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Naive Bayes Model Accuracy: {accuracy}")

Naive Bayes Model Accuracy: 0.5454545454545454


In [ ]:
# --- Naive Bayes ---
import joblib
joblib.dump(model, 'models/naive_bayes_model.joblib')
print("Saved naive_bayes_model.joblib")

Saved naive_bayes_model.joblib
